In [ ]:
import os, json, time, random, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix, classification_report,
    matthews_corrcoef, cohen_kappa_score
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF:", tf.__version__)
print("Keras:", keras.__version__)


In [ ]:
WISDM_ROOT = r"E:\WISDM_ar_v1.1"
WISDM_FILE = os.path.join(WISDM_ROOT, "WISDM_ar_v1.1_raw.txt")  # change if your file name differs

RUN_NAME = "WISDM_TRANSFORMER_STREAMLINED"
OUT_DIR = os.path.join(WISDM_ROOT, "OUTPUT_WISDM_TRANSFORMER", RUN_NAME)

DIRS = {
    "conf_mats": os.path.join(OUT_DIR, "confusion_matrices"),
    "models":    os.path.join(OUT_DIR, "models"),
    "plots":     os.path.join(OUT_DIR, "plots"),
    "reports":   os.path.join(OUT_DIR, "reports"),
    "tflite":    os.path.join(OUT_DIR, "tflite"),
}
for d in DIRS.values():
    os.makedirs(d, exist_ok=True)

print("WISDM_FILE:", WISDM_FILE)
print("OUT_DIR:", OUT_DIR)


In [ ]:
# Standard WISDM activities (common 6-class subset)
LABELS = ["Downstairs", "Jogging", "Sitting", "Standing", "Upstairs", "Walking"]

# Map strings in raw file to indices
label2idx = {lab: i for i, lab in enumerate(LABELS)}
idx2label = {i: lab for lab, i in label2idx.items()}

NUM_CLASSES = len(LABELS)
print("Classes:", LABELS)


In [ ]:
def plot_cm_values(cm, class_names, title, save_path, normalize=True):
    cm_plot = cm.astype(np.float32)
    if normalize:
        cm_plot = cm_plot / np.maximum(cm_plot.sum(axis=1, keepdims=True), 1e-9)

    plt.figure(figsize=(8.6, 7.6))
    plt.imshow(cm_plot, interpolation="nearest")
    plt.title(title, fontsize=13)
    plt.colorbar(fraction=0.046, pad=0.04)

    ticks = np.arange(len(class_names))
    plt.xticks(ticks, class_names, rotation=45, ha="right", fontsize=9)
    plt.yticks(ticks, class_names, fontsize=9)

    fmt = ".2f" if normalize else "d"
    thresh = cm_plot.max() * 0.55

    for i in range(cm_plot.shape[0]):
        for j in range(cm_plot.shape[1]):
            v = cm_plot[i, j]
            plt.text(j, i, format(v, fmt),
                     ha="center", va="center",
                     fontsize=9, fontweight="bold",
                     color="white" if v > thresh else "black")

    plt.ylabel("True label")
    plt.xlabel("Predicted label")
    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.close()


In [ ]:
def load_wisdm_raw(path):
    # Read lines (WISDM has trailing ';')
    with open(path, "r") as f:
        lines = f.readlines()

    rows = []
    for line in lines:
        line = line.strip()
        if not line:
            continue
        # remove trailing ';'
        if line.endswith(";"):
            line = line[:-1]
        parts = line.split(",")
        if len(parts) != 6:
            continue

        user, act, ts, x, y, z = parts
        rows.append([user, act, ts, x, y, z])

    df = pd.DataFrame(rows, columns=["user", "activity", "timestamp", "x", "y", "z"])

    # Coerce numeric
    for c in ["timestamp", "x", "y", "z"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Clean
    df.dropna(inplace=True)
    df = df[df["timestamp"] > 0]
    df = df[df["activity"].isin(label2idx)]  # keep only 6-class subset
    df.reset_index(drop=True, inplace=True)

    # Sort by user/time (important for windowing)
    df.sort_values(["user", "timestamp"], inplace=True)
    df.reset_index(drop=True, inplace=True)

    return df

df = load_wisdm_raw(WISDM_FILE)
print("Loaded:", df.shape)
print(df.head())
print("Activities present:", sorted(df["activity"].unique()))


In [ ]:
WIN  = 80
STEP = 40  # 50% overlap

FEATURES = ["x", "y", "z"]

def make_windows_userwise(df, win=WIN, step=STEP):
    Xw, yw = [], []
    for user_id, sub in df.groupby("user", sort=False):
        X = sub[FEATURES].values.astype(np.float32)
        y = sub["activity"].map(label2idx).values.astype(np.int32)

        for i in range(0, len(X) - win + 1, step):
            xs = X[i:i+win]
            ys = y[i:i+win]
            lab = np.bincount(ys).argmax()
            Xw.append(xs)
            yw.append(lab)

    return np.array(Xw, dtype=np.float32), np.array(yw, dtype=np.int32)

Xw, yw = make_windows_userwise(df)
print("Windows:", Xw.shape, "Labels:", yw.shape)
print("Class counts:", np.bincount(yw))


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    Xw, yw, test_size=0.2, random_state=SEED, stratify=yw
)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train.reshape(-1, X_train.shape[-1])).reshape(X_train.shape)
X_test_s  = scaler.transform(X_test.reshape(-1, X_test.shape[-1])).reshape(X_test.shape)

np.save(os.path.join(OUT_DIR, "X_train_s.npy"), X_train_s)
np.save(os.path.join(OUT_DIR, "X_test_s.npy"),  X_test_s)
np.save(os.path.join(OUT_DIR, "y_train.npy"),   y_train)
np.save(os.path.join(OUT_DIR, "y_test.npy"),    y_test)

with open(os.path.join(OUT_DIR, "labels.json"), "w") as f:
    json.dump({"labels": LABELS, "label2idx": label2idx}, f, indent=2)



In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
import keras as K
import numpy as np

# --- Keras 3 safe learned positional embedding ---
class AddPositionEmbedding(layers.Layer):
    def __init__(self, max_len, d_model, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.d_model = d_model
        self.pos_emb = layers.Embedding(input_dim=max_len, output_dim=d_model)

    def call(self, x):
        # x: (B, T, D)
        T = K.ops.shape(x)[1]  # Keras-safe dynamic shape
        pos = K.ops.arange(0, T, dtype="int32")  # (T,)
        pe = self.pos_emb(pos)                  # (T, D)
        pe = K.ops.expand_dims(pe, axis=0)      # (1, T, D)
        return x + pe                           # broadcast over batch

def encoder(x, d_model, num_heads, mlp_dim, dropout):
    attn = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=d_model // num_heads,
        dropout=dropout
    )(x, x)
    x = layers.LayerNormalization(epsilon=1e-6)(x + attn)

    ff = layers.Dense(mlp_dim, activation="gelu")(x)
    ff = layers.Dropout(dropout)(ff)
    ff = layers.Dense(d_model)(ff)
    ff = layers.Dropout(dropout)(ff)

    return layers.LayerNormalization(epsilon=1e-6)(x + ff)

def build_transformer(input_shape, num_classes,
                      patch_len=8, d_model=64, depth=2,
                      num_heads=4, mlp_dim=128,
                      dropout=0.15, lr=1e-3,
                      max_len=2048):

    inp = keras.Input(shape=input_shape)  # (WIN, n_features)

    # Patch embedding (stride = patch_len)
    x = layers.Conv1D(
        filters=d_model,
        kernel_size=patch_len,
        strides=patch_len,
        padding="valid"
    )(inp)

    # Add learned positional embedding safely (Keras 3)
    x = AddPositionEmbedding(max_len=max_len, d_model=d_model)(x)
    x = layers.Dropout(dropout)(x)

    for _ in range(depth):
        x = encoder(x, d_model, num_heads, mlp_dim, dropout)

    x = layers.GlobalAveragePooling1D()(x)
    out = layers.Dense(num_classes, activation="softmax")(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


# ---- Fixed hyperparameters (edit here only) ----
HP = dict(
    patch_len=8,
    d_model=64,
    depth=2,
    num_heads=4,
    mlp_dim=128,
    dropout=0.15,
    lr=1e-3,
)

best_model = build_transformer(
    input_shape=(WIN, X_train_s.shape[-1]),
    num_classes=NUM_CLASSES,
    **HP
)
best_model.summary()


In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", patience=3, factor=0.5, min_lr=1e-5),
]

history = best_model.fit(
    X_train_s, y_train,
    validation_split=0.15,
    epochs=40,
    batch_size=128,
    callbacks=callbacks,
    verbose=1
)

keras_path = os.path.join(DIRS["models"], "transformer_best_fp32.keras")
best_model.save(keras_path)
print(" Saved:", keras_path)


In [ ]:
probs = best_model.predict(X_test_s, batch_size=256, verbose=0)
y_pred = np.argmax(probs, axis=1)

cm = confusion_matrix(y_test, y_pred)
cm_path = os.path.join(DIRS["conf_mats"], "TF_TRANSFORMER_FP32_cm_norm.png")
plot_cm_values(cm, LABELS, "TF Transformer FP32 (WISDM) - Normalized CM", cm_path, normalize=True)

report = classification_report(y_test, y_pred, target_names=LABELS)
with open(os.path.join(DIRS["reports"], "TF_TRANSFORMER_FP32_report.txt"), "w") as f:
    f.write(report)

acc  = accuracy_score(y_test, y_pred)
bacc = balanced_accuracy_score(y_test, y_pred)
p_m, r_m, f1_m, _ = precision_recall_fscore_support(y_test, y_pred, average="macro", zero_division=0)
p_w, r_w, f1_w, _ = precision_recall_fscore_support(y_test, y_pred, average="weighted", zero_division=0)
mcc   = matthews_corrcoef(y_test, y_pred)
kappa = cohen_kappa_score(y_test, y_pred)

df_base = pd.DataFrame([{
    "model": "TF_TRANSFORMER_FP32",
    "accuracy": float(acc),
    "balanced_accuracy": float(bacc),
    "precision_macro": float(p_m),
    "recall_macro": float(r_m),
    "f1_macro": float(f1_m),
    "precision_weighted": float(p_w),
    "recall_weighted": float(r_w),
    "f1_weighted": float(f1_w),
    "mcc": float(mcc),
    "kappa": float(kappa),
}])

base_csv = os.path.join(OUT_DIR, "summary_transformer_fp32.csv")
df_base.to_csv(base_csv, index=False)

print(" Saved CM + report + baseline summary CSV")
print(df_base)


In [ ]:
_ = best_model.predict(X_test_s[:1], verbose=0)

SAVEDMODEL_DIR = os.path.join(DIRS["models"], "transformer_savedmodel")
if os.path.exists(SAVEDMODEL_DIR):
    shutil.rmtree(SAVEDMODEL_DIR)

best_model.export(SAVEDMODEL_DIR)
print(" Exported SavedModel:", SAVEDMODEL_DIR)
